In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "prajjwal1/bert-tiny"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

In [3]:
train['label'] = train['label'].map({'positive': 1, 'negative': 0})

In [6]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train['review'],
    train['label'],
    test_size=0.2,
    random_state=42
)

In [7]:
def tokenize_function(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256
    )

train_encodings = tokenize_function(train_texts.tolist())
val_encodings = tokenize_function(val_texts.tolist())
test_encodings = tokenize_function(test['review'].tolist())

In [8]:
class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

In [9]:
from torch.utils.data import DataLoader

train_dataset = IMDBDataset(train_encodings, train_labels.tolist())
val_dataset = IMDBDataset(val_encodings, val_labels.tolist())
test_dataset = IMDBDataset(test_encodings)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

In [10]:
from sklearn.metrics import f1_score

def evaluate(model, data_loader, device):
    model.eval()
    
    correct = 0
    total = 0
    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(data_loader)
    accuracy = correct / total
    f1 = f1_score(all_labels, all_preds, average='macro')

    return avg_loss, accuracy, f1

In [11]:
from tqdm import tqdm

device = torch.device("cpu")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

for epoch in range(5):
    
    print(f"Epoch {epoch}")
    
    model.train()
    
    total_loss = 0
    
    for batch in tqdm(train_loader):
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print("Loss:", total_loss / len(train_loader))
    val_loss, val_acc, val_f1 = evaluate(model, val_loader, device)
    
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")
    print(f"Validation F1: {val_f1:.4f}")

Epoch 0


  0%|                                                  | 0/2250 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|███████████████████████████████████████| 2250/2250 [09:37<00:00,  3.90it/s]


Loss: 0.4166206059737338
Validation Loss: 0.3384
Validation Accuracy: 0.8527
Validation F1: 0.8519
Epoch 1


100%|███████████████████████████████████████| 2250/2250 [09:47<00:00,  3.83it/s]


Loss: 0.2891187835865551
Validation Loss: 0.3076
Validation Accuracy: 0.8702
Validation F1: 0.8697
Epoch 2


...

Loss: 0.2260564012395011
Validation Loss: 0.3110
Validation Accuracy: 0.8779
Validation F1: 0.8778
Epoch 3


100%|███████████████████████████████████████| 2250/2250 [09:04<00:00,  4.13it/s]


Loss: 0.17095744527462456
Validation Loss: 0.3395
Validation Accuracy: 0.8800
Validation F1: 0.8800
Epoch 4


100%|███████████████████████████████████████| 2250/2250 [09:41<00:00,  3.87it/s]


Loss: 0.12883676573696237
Validation Loss: 0.3446
Validation Accuracy: 0.8811
Validation F1: 0.8811


In [23]:
# model.eval()

# correct = 0
# total = 0

# with torch.no_grad():
#     for batch in val_loader:
#         input_ids = batch['input_ids'].to(device)
#         attention_mask = batch['attention_mask'].to(device)
#         labels = batch['labels'].to(device)

#         outputs = model(input_ids=input_ids, attention_mask=attention_mask)
#         preds = torch.argmax(outputs.logits, dim=1)

#         correct += (preds == labels).sum().item()
#         total += labels.size(0)

# print("Validation Accuracy:", correct / total)

Validation Accuracy: 0.8793333333333333


In [15]:
# test_loss, test_acc, test_f1 = evaluate(model, test_loader, device)

# print("\nFinal Test Results:")
# print(f"Test Loss: {test_loss:.4f}")
# print(f"Test Accuracy: {test_acc:.4f}")
# print(f"Test F1: {test_f1:.4f}")

In [13]:
preds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        batch_preds = torch.argmax(outputs.logits, dim=1)

        preds.extend(batch_preds.cpu().numpy())

print(len(preds))

Predicting: 100%|█████████████████████████████| 313/313 [00:17<00:00, 18.14it/s]

5000


In [14]:
pred_labels = ['positive' if p == 1 else 'negative' for p in preds]

submission = pd.DataFrame({
    'Id': test['Id'],
    'Label': pred_labels
})

submission.to_csv("prediction.csv", index=False)